# Pre-Canonical Inscriptions (NB 61)

Browser for the 6 inscriptions tagged `pre_canonical` in `data/quipu_data.csv`. After the May 2026 lenient title-region relaxation, only these structurally-divergent inscriptions remain pre-canonical — see `memory/pre_canonical_inscriptions.md` for the full record of what got reclassified as canonical.

Categories surfaced:
- **Non-structural image** (1) — Sparkle Magical Cat, no 12-byte structural header
- **Encrypted with non-canonical tone bytes** (4) — pre-tone-canonization test pairs
- **Pre-redesign celestial** (1) — old Sky of al-Jawza with kind=0x81 bit-packed flags

## Setup

In [1]:
import os, sys, json
REPO = os.path.abspath('..')
sys.path.insert(0, REPO)
sys.path.insert(0, os.path.join(REPO, 'canonical'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv(os.path.join(REPO, 'data/quipu_data.csv'))
pre = df[df['canonical_status'] == 'pre_canonical'].copy()
print(f'{len(pre)} pre-canonical inscriptions:')
pre[['root_txid','type_name','label','title','total_bytes','blockheight']]

14 pre-canonical inscriptions:


,root_txid,type_name,label,title,total_bytes,blockheight
0,a2e9f2ebe1ceeae368b734fcd861549c813d5cfb009093...,image,apocrypha,NaN,2611,4221666
1,c1542c10399a09a5471799133c472a6652d5cb3b7b421e...,image,apocrypha,Sparkle🐈‍⬛MagicalCat🐈‍⬛✨💜Fovever💜✨,2630,4222137
2,a01e8625d653f4a8686b5b9e20ca653e59ebcd8bf2ca2a...,image,apocrypha,Peter Bea,7701,4224224
3,9e42c7ab6f47dadc0c36d22fb20f69c9db0daa5c719cd8...,image,apocrypha,This was Peter on her blanket during a ride to...,15472,4240897
6,d68175766b70f7163aec93e5a4e81480a6c6dd51d05773...,encrypted,apocrypha,NaN,2800,4251800
7,d0209a0f85872d6826c58bc23fab37c8b21feb22c15a5a...,encrypted,apocrypha,NaN,15655,4252445
8,89b51b4852b0e80f49cdb229d85ef4757d943c9fe4ba62...,encrypted,apocrypha,NaN,112,4267974
9,f278e466012fb78422834742c6440c935f4cc2ef64e722...,encrypted,apocrypha,NaN,112,4270461
10,aa0c3ea6b38b2238725df2457eb328bbebd7d91ac5241c...,image,apocrypha,Dr. Doeg en Buenos Aires|G� ��9�M��X+�:4��vךGe...,41030,4289591
16,33a40fef2cec220c694a35f591b1d992d8d75bfe5b8bdc...,image,ca,NaN,163852,4510396


In [2]:
# Helper — load the body bytes for a row
def load_blob(row):
    return open(os.path.join(REPO, 'data', row['body_file']), 'rb').read()

# Helper — render a v1-shaped image quipu (with back-computed body offset)
def render_image_v1ish(row, ax=None):
    blob = load_blob(row)
    dims = json.loads(row['dimensions_json'])
    W, H, color, bd = dims['W'], dims['H'], dims['color'], dims['bit_depth']
    ch = 1 if color == 0 else 3
    expected_body = (W * H * ch * bd + 7) // 8
    body_offset = len(blob) - expected_body
    body = blob[body_offset:body_offset + expected_body]
    # Unpack MSB-first into a flat list of values
    bits = []
    for byte in body:
        for i in range(7, -1, -1):
            bits.append((byte >> i) & 1)
    total_vals = W * H * ch
    values = []
    for i in range(total_vals):
        v = 0
        for j in range(bd):
            v = (v << 1) | bits[i*bd + j]
        values.append(v)
    max_val = (1 << bd) - 1
    arr = np.array(values, dtype=np.float32) / max_val
    if ch == 1:
        arr = arr.reshape((H, W))
        cmap = 'gray'
    else:
        arr = arr.reshape((H, W, 3))
        cmap = None
    if ax is None:
        fig, ax = plt.subplots(figsize=(4, 4*H/W if W else 4))
    ax.imshow(arr, cmap=cmap)
    ax.axis('off')
    return ax

## Category A — Non-structural image (1)

Magic + type byte ✓ but no 12-byte structural header — title text starts immediately after byte 5.

In [3]:
row = df[df['root_txid'].str.startswith('a2e9f2eb')].iloc[0]
blob = load_blob(row)
print(f'txid: {row["root_txid"]}')
print(f'label: {row["label"]}  blockheight: {row["blockheight"]}')
print(f'total bytes: {len(blob)}')
print(f'bytes[0:6] (magic + type + tone): {blob[:6].hex()}  — c1dd0001 03 ff')
print(f'bytes[6:60] (where structural header should be, but isn\'t):')
print(f'  {blob[6:60]!r}')
print()
print('Interpretation: the bytes that v1 spec reserves for color/W/H/bit_depth are actually title text:')
print(f'  byte 6 = 0x{blob[6]:02x} (not 0x00 or 0x01)')
print(f'  bytes 7-11 = {blob[7:12]!r}  (continuation of title text)')
# The title text continues; try to find where pixel-like data starts
title_attempt = blob[5:80].split(b'\n')[0:3]
print(f'\nFirst few title lines: {title_attempt}')

txid: a2e9f2ebe1ceeae368b734fcd861549c813d5cfb009093237f86b71f0048bda7
label: apocrypha  blockheight: 4221666
total bytes: 2611
bytes[0:6] (magic + type + tone): c1dd000103ff  — c1dd0001 03 ff
bytes[6:60] (where structural header should be, but isn't):
  b'  \x05Sparkle\nMagical Cat\n\xe2\x9c\xa8\xf0\x9f\x92\x9cFovever\xf0\x9f\x92\x9c\xe2\x9c\xa8\n\xe6\xf7\xbd\xef{\xde\xf7\xad\xef'

Interpretation: the bytes that v1 spec reserves for color/W/H/bit_depth are actually title text:
  byte 6 = 0x20 (not 0x00 or 0x01)
  bytes 7-11 = b' \x05Spa'  (continuation of title text)

First few title lines: [b'\xff  \x05Sparkle', b'Magical Cat', b'\xe2\x9c\xa8\xf0\x9f\x92\x9cFovever\xf0\x9f\x92\x9c\xe2\x9c\xa8']


## Category D — Encrypted with non-canonical tone bytes (4)

Apocrypha encrypted-quipu test pairs from before tone was canonized to {0x00, 0x01, 0xff}. These used 0x03 and 0x0e as experimental tone values. The structural header is otherwise compatible with the 0x0e family. Encrypted bodies aren't decrypted here — keys are not in scope for this view.

In [ ]:
old_tones = ['d6817576', 'd0209a0f', '89b51b48', 'f278e466']
for prefix in old_tones:
    row = df[df['root_txid'].str.startswith(prefix)].iloc[0]
    blob = load_blob(row)
    sub_family = blob[6] if len(blob) > 6 else None
    variant = blob[7] if len(blob) > 7 else None
    sf_name = {0xae:'aes', 0xec:'ecies', 0x0d:'drop'}.get(sub_family, f'unknown_0x{sub_family:02x}')
    print(f'=== {prefix}… ===')
    print(f'  label: {row["label"]}  block: {row["blockheight"]}')
    print(f'  bytes[0:8]: {blob[:8].hex()}')
    print(f'  type:        0x0e (encrypted)')
    print(f'  tone:        0x{blob[5]:02x}  ⚠ non-canonical (canonical: 0x00, 0x01, 0xff)')
    print(f'  sub_family:  0x{sub_family:02x} ({sf_name})')
    print(f'  variant:     0x{variant:02x}')
    print(f'  total bytes: {len(blob)}')
    print()

## Category E — Pre-redesign celestial (1)

The old Sky of al-Jawza inscription used the original bit-packed celestial format (kind byte with high-bit flags) before the May 2026 redesign that split kind/grouped/meta into separate bytes. Superseded by the canonical `2ae7fe90…` inscription.

In [ ]:
row = df[df['root_txid'].str.startswith('4e53bb26')].iloc[0]
blob = load_blob(row)
print(f'txid: {row["root_txid"]}')
print(f'label: {row["label"]}  block: {row["blockheight"]}')
print(f'total bytes: {len(blob)}')
print(f'bytes[0:12]: {blob[:12].hex()}')
print(f'  c1dd0001 — magic')
print(f'  ce       — type (celestial)')
print(f'  ff       — tone (reverence)')
print(f'  byte 6 = 0x{blob[6]:02x} = 0b{blob[6]:08b}  ⚠ pre-redesign bit-packed kind/flags')
print(f'  (canonical v1 uses separate kind/grouped/meta bytes)')
print()
print(f'First 200 bytes of body:')
print(f'  {blob[7:200]!r}')

## Estandarte v1 implications

Per `memory/pre_canonical_inscriptions.md`:

- 25 canonical_v1 inscriptions are available as canonical-example citations for Estandarte v1 (incl. the 8 reclassified by the lenient title-region rule).
- The 6 inscriptions shown above remain pre-canonical: 1 non-structural image, 4 encrypted with non-canonical tone bytes, 1 pre-redesign celestial. These are not cited as canonical examples.
- Estandarte v1 should register `0x0c` (the pre-canonical hash-cert that became 0xcc subtype 0x0001) with `status=3 deprecated`. That inscription (20db7d45) is in `data/quipu_data.csv` tagged `not_yet_canonicalized`.
- The non-structural 0x03 variant (a2e9f2eb) and the old celestial kind layout don't need their own Estandarte registrations — they're individual experiments, not separate types.